# Portfolio and Risk Analysis

This notebook performs the final portfolio comparison across the best available strategies from MLP, FT-Transformer, Temporal, and the XGBoost baseline. It uses persisted backtest outputs only and reports validation selection, test performance, alpha/beta risk, turnover, and portfolio interpretation.

## Setup

The portfolio analysis uses the persisted backtest outputs from notebook 11. To keep the final comparison implementable and comparable, every ranking, selection, table, and plot in this notebook is filtered to a single cost assumption: 25 bps one-way transaction cost.


In [ ]:
from pathlib import Path
import os

Path("/tmp/matplotlib-cache").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "outputs").exists() and (ROOT / "ML_For_Finance_Project-AxelTurinPlessia-362559-ClementMeddeb-346164").exists():
    ROOT = ROOT / "ML_For_Finance_Project-AxelTurinPlessia-362559-ClementMeddeb-346164"

BACKTEST_DIR = ROOT / "outputs" / "backtests" / "dl_xgb_score_strategies"
TABLE_DIR = ROOT / "outputs" / "tables"
FIGURE_DIR = ROOT / "outputs" / "figures"
PREDICTION_DIR = ROOT / "outputs" / "predictions"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_COST_BPS = 25.0

performance_all = pd.read_csv(BACKTEST_DIR / "dl_xgb_score_strategy_performance_summary.csv")
monthly_returns_all = pd.read_csv(BACKTEST_DIR / "dl_xgb_score_strategy_monthly_returns.csv", parse_dates=["month"])
predictions = pd.read_parquet(PREDICTION_DIR / "dl_xgb_predictions.parquet")
predictions["MthCalDt"] = pd.to_datetime(predictions["MthCalDt"])

performance = performance_all.loc[np.isclose(performance_all["one_way_cost_bps"].astype(float), ANALYSIS_COST_BPS)].copy()
monthly_returns = monthly_returns_all.loc[np.isclose(monthly_returns_all["one_way_cost_bps"].astype(float), ANALYSIS_COST_BPS)].copy()

if performance.empty or monthly_returns.empty:
    raise ValueError(f"No portfolio backtest rows found at {ANALYSIS_COST_BPS:.0f} bps")

print(
    f"Loaded {len(performance)} performance rows and {len(monthly_returns)} monthly return rows "
    f"at {ANALYSIS_COST_BPS:.0f} bps from {ROOT}"
)


## 1. Strategy Universe, Return Definition, and Heatmap

All portfolio metrics below use the notebook 11 backtest artifacts filtered to `one_way_cost_bps = 25`. This removes the previous mixing of 0 bps, 10 bps, and 25 bps strategies.

Notebook 11 constructs monthly weights and returns as follows. For a month, `gross_return = sum_i weight_i * realized_return_i`. Turnover is the sum of absolute changes in PERMNO-level weights versus the previous month; in the first month it is the absolute gross exposure opened. Transaction cost is `turnover * (one_way_cost_bps / 10000)`. Therefore `net_return = gross_return - transaction_cost`, and notebook 15 uses `net_return` for alpha, beta, and wealth curves.

Weight normalization depends on the rule. `long_short` uses long exposure 1.0 and short exposure 1.0, so it is approximately dollar neutral with gross exposure 2.0. `gross_normalized_long_short` uses long exposure 0.5 and short exposure 0.5, so it is dollar neutral with gross exposure 1.0. `long_short_130_30` uses long exposure 1.3 and short exposure 0.3, so it has gross exposure 1.6 and net exposure 1.0. The analysis table keeps `average_gross_exposure` and `average_net_exposure` so the exposure convention is visible.


In [ ]:
SPEC_COLS = ["score_label", "rule", "q", "sign_gate", "threshold_gate", "zero_threshold", "one_way_cost_bps"]
REQUIRED_FAMILIES = ["MLP", "FT", "Temporal", "baseline_model"]

def model_family(score_label):
    label = str(score_label)
    if label.startswith("mlp"):
        return "MLP"
    if label.startswith("ft"):
        return "FT"
    if label.startswith("temporal"):
        return "Temporal"
    if label.startswith("xgb"):
        return "baseline_model"
    return "Other"

def strategy_name(row):
    gate = "thr" if bool(row.get("threshold_gate", False)) else "plain"
    return f"{row['model_family']} | {row['score_label']} | {row['rule']} | {gate}={row['zero_threshold']:.2f} | cost={row['one_way_cost_bps']:.0f}bps"

def enforce_family_coverage(ranked, required_families, n=5):
    selected = ranked.head(n).copy()
    for family in required_families:
        if selected["model_family"].eq(family).any():
            continue
        candidate = ranked.loc[ranked["model_family"].eq(family)].head(1)
        if candidate.empty:
            raise ValueError(f"No eligible {family} strategy found at {ANALYSIS_COST_BPS:.0f} bps")
        protected = set(required_families) - {family}
        replace_pool = selected.loc[~selected["model_family"].isin(protected)].sort_values("sharpe")
        if replace_pool.empty:
            replace_pool = selected.sort_values("sharpe")
        drop_idx = replace_pool.index[0]
        selected = pd.concat([selected.drop(index=drop_idx), candidate], ignore_index=False)
        selected = selected.drop_duplicates(subset=SPEC_COLS).sort_values("sharpe", ascending=False).head(n)
    return selected.sort_values("sharpe", ascending=False).copy()

validation = performance.loc[performance["split"].eq("validation")].copy()
validation["model_family"] = validation["score_label"].map(model_family)
validation = validation.loc[validation["model_family"].isin(REQUIRED_FAMILIES)].copy()
validation["strategy"] = validation.apply(strategy_name, axis=1)

ranking = validation.sort_values("sharpe", ascending=False).copy()
ranking["selection_universe"] = f"validation Sharpe, {ANALYSIS_COST_BPS:.0f}bps only"
ranking.to_csv(TABLE_DIR / "portfolio_validation_strategy_ranking_all_models.csv", index=False)

selected = enforce_family_coverage(ranking, REQUIRED_FAMILIES, n=5)
selected["selection_reason"] = np.where(
    selected.index.isin(ranking.head(5).index),
    f"top 5 validation Sharpe at {ANALYSIS_COST_BPS:.0f}bps",
    f"added to ensure model-family coverage at {ANALYSIS_COST_BPS:.0f}bps",
)
selected.to_csv(TABLE_DIR / "portfolio_selected_top5_strategies.csv", index=False)

metric_cols = ["sharpe", "annualized_return", "annualized_volatility", "max_drawdown", "average_turnover"]
heatmap_table = selected.set_index("strategy")[metric_cols].rename(columns={
    "annualized_return": "return",
    "annualized_volatility": "volatility",
    "max_drawdown": "drawdown",
    "average_turnover": "turnover",
})
heatmap_table.to_csv(TABLE_DIR / "portfolio_top5_validation_heatmap_table.csv")

scaled = heatmap_table.copy()
for col in scaled.columns:
    denom = scaled[col].std(ddof=0)
    scaled[col] = 0.0 if denom == 0 or np.isnan(denom) else (scaled[col] - scaled[col].mean()) / denom

fig, ax = plt.subplots(figsize=(10, 4.8))
im = ax.imshow(scaled.to_numpy(), cmap="coolwarm", aspect="auto")
ax.set_yticks(np.arange(len(scaled.index)))
ax.set_yticklabels(scaled.index)
ax.set_xticks(np.arange(len(scaled.columns)))
ax.set_xticklabels(scaled.columns, rotation=25, ha="right")
for i in range(heatmap_table.shape[0]):
    for j, col in enumerate(heatmap_table.columns):
        ax.text(j, i, f"{heatmap_table.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title(f"Selected top strategies: validation metrics ({ANALYSIS_COST_BPS:.0f}bps net)")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "portfolio_top5_validation_metric_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()
heatmap_table


## 2. Validation Ranking and Test Performance

The top-ten table is ranked by validation Sharpe after filtering to 25 bps. Test performance is joined only after the validation ranking and top-five selection are fixed, preserving the train/validation/test discipline.


In [ ]:
top10_validation = ranking.head(10).copy()
top10_validation.to_csv(TABLE_DIR / "portfolio_top10_validation_sharpe.csv", index=False)
print(f"Top 10 validation strategies by Sharpe at {ANALYSIS_COST_BPS:.0f} bps")
print(top10_validation[["strategy", "sharpe", "annualized_return", "annualized_volatility", "max_drawdown", "average_turnover", "average_gross_exposure", "average_net_exposure"]].to_string(index=False))

test = performance.loc[performance["split"].eq("test")].copy()
test["model_family"] = test["score_label"].map(model_family)
test["strategy"] = test.apply(strategy_name, axis=1)
selected_test = selected[SPEC_COLS + ["strategy", "model_family", "selection_reason"]].merge(
    test, on=SPEC_COLS, how="left", suffixes=("_selected", "")
)
selected_test["strategy"] = selected_test["strategy_selected"].fillna(selected_test["strategy"])
selected_test["model_family"] = selected_test["model_family_selected"].fillna(selected_test["model_family"])
selected_test = selected_test.drop(columns=[c for c in ["strategy_selected", "model_family_selected"] if c in selected_test.columns])
selected_test.to_csv(TABLE_DIR / "portfolio_selected_top5_test_performance.csv", index=False)
print("\nSelected top 5 test performance")
print(selected_test[["strategy", "sharpe", "annualized_return", "annualized_volatility", "max_drawdown", "average_turnover", "average_gross_exposure", "average_net_exposure"]].to_string(index=False))


## 3. Selected Top 5 Strategies

The selected set is the top five validation-Sharpe strategies after the 25 bps filter, with a guard that requires at least one MLP, FT, Temporal, and baseline strategy. In this run, the top five already satisfy that coverage, so no lower-ranked replacement is needed.


In [ ]:
selected_display = selected[[
    "strategy", "model_family", "score_label", "rule", "zero_threshold", "one_way_cost_bps",
    "sharpe", "annualized_return", "max_drawdown", "average_turnover",
    "average_gross_exposure", "average_net_exposure", "selection_reason"
]]
print(selected_display.to_string(index=False))
selected_display


## 4. Alpha / Beta Analysis

For each selected strategy, monthly net portfolio returns are regressed on a market return proxy:

`R_portfolio = alpha + beta * R_market + epsilon`

If a CRSP value-weighted market return is available in the prediction panel, it is used. Otherwise, the market return is the cross-sectional average of realized one-month stock returns each month. This fallback is appropriate here because the portfolios and the proxy are formed from the same equity universe; it captures the common market movement faced by the backtested strategies when an external market index is not stored in the artifacts. Alpha is annualized from the monthly intercept.


In [ ]:
market_candidates = ["vwretd", "vwretx", "mkt_ret", "market_return", "crsp_vwret"]
market_col = next((col for col in market_candidates if col in predictions.columns), None)
if market_col is not None:
    market_returns = predictions.loc[predictions["split"].eq("test"), ["MthCalDt", market_col]].dropna()
    market_returns = market_returns.groupby("MthCalDt", as_index=False)[market_col].mean().rename(columns={"MthCalDt": "month", market_col: "market_return"})
    market_definition = f"CRSP value-weighted market return from `{market_col}`"
else:
    market_returns = predictions.loc[predictions["split"].eq("test"), ["MthCalDt", "target_ret_1m"]].dropna()
    market_returns = market_returns.groupby("MthCalDt", as_index=False)["target_ret_1m"].mean().rename(columns={"MthCalDt": "month", "target_ret_1m": "market_return"})
    market_definition = "equal-weight cross-sectional average realized return proxy"

print("Market return definition:", market_definition)
market_returns.to_csv(TABLE_DIR / "portfolio_market_return_proxy.csv", index=False)

def spec_mask(df, spec):
    mask = df["split"].eq("test")
    for col in SPEC_COLS:
        if col in ["q", "zero_threshold", "one_way_cost_bps"]:
            mask &= np.isclose(df[col].astype(float), float(spec[col]), equal_nan=True)
        else:
            mask &= df[col].eq(spec[col])
    return mask

def ols_alpha_beta(portfolio_returns, market_returns):
    reg = portfolio_returns.merge(market_returns, on="month", how="inner").dropna(subset=["net_return", "market_return"])
    y = reg["net_return"].to_numpy(dtype=float)
    x = reg["market_return"].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    coef = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ coef
    n, k = X.shape
    sigma2 = float((resid @ resid) / max(n - k, 1))
    cov = sigma2 * np.linalg.inv(X.T @ X)
    se_alpha = float(np.sqrt(cov[0, 0])) if cov[0, 0] >= 0 else np.nan
    t_alpha = float(coef[0] / se_alpha) if se_alpha and not np.isnan(se_alpha) else np.nan
    return float(coef[0] * 12.0), float(coef[1]), t_alpha, len(reg)

alpha_rows = []
selected_monthly_frames = []
for _, spec in selected.iterrows():
    part = monthly_returns.loc[spec_mask(monthly_returns, spec)].copy().sort_values("month")
    if part.empty:
        continue
    part["strategy"] = spec["strategy"]
    selected_monthly_frames.append(part)
    alpha, beta, t_alpha, nobs = ols_alpha_beta(part[["month", "net_return"]], market_returns)
    test_match = pd.Series(True, index=selected_test.index)
    for col in SPEC_COLS:
        if col in ["q", "zero_threshold", "one_way_cost_bps"]:
            test_match &= np.isclose(selected_test[col].astype(float), float(spec[col]), equal_nan=True)
        else:
            test_match &= selected_test[col].eq(spec[col])
    test_row = selected_test.loc[test_match].head(1)
    alpha_rows.append({
        "strategy": spec["strategy"],
        "model_family": spec["model_family"],
        "alpha": alpha,
        "beta": beta,
        "tstat_alpha": t_alpha,
        "n_months": nobs,
        "sharpe": float(test_row["sharpe"].iloc[0]) if not test_row.empty else np.nan,
        "return": float(test_row["annualized_return"].iloc[0]) if not test_row.empty else np.nan,
        "vol": float(test_row["annualized_volatility"].iloc[0]) if not test_row.empty else np.nan,
        "drawdown": float(test_row["max_drawdown"].iloc[0]) if not test_row.empty else np.nan,
        "turnover": float(test_row["average_turnover"].iloc[0]) if not test_row.empty else np.nan,
    })

alpha_beta = pd.DataFrame(alpha_rows).sort_values("alpha", ascending=False)
alpha_beta.to_csv(TABLE_DIR / "portfolio_alpha_beta_top5.csv", index=False)
print(alpha_beta.to_string(index=False))

final_table = selected_test[["strategy", "sharpe", "annualized_return", "annualized_volatility", "max_drawdown", "average_turnover"]].rename(columns={
    "annualized_return": "return",
    "annualized_volatility": "vol",
    "max_drawdown": "drawdown",
    "average_turnover": "turnover",
})
final_table = final_table.merge(alpha_beta[["strategy", "alpha", "beta", "tstat_alpha"]], on="strategy", how="left")
final_table = final_table[["strategy", "sharpe", "return", "vol", "alpha", "beta", "tstat_alpha", "drawdown", "turnover"]]
final_table.to_csv(TABLE_DIR / "portfolio_final_table_25bps.csv", index=False)
print("\nFinal 25 bps portfolio table")
print(final_table.to_string(index=False))

selected_monthly = pd.concat(selected_monthly_frames, ignore_index=True) if selected_monthly_frames else pd.DataFrame()
selected_monthly.to_csv(TABLE_DIR / "portfolio_selected_top5_monthly_returns.csv", index=False)
final_table


In [ ]:
if not selected_monthly.empty:
    cumulative = selected_monthly.sort_values(["strategy", "month"]).copy()
    cumulative["wealth"] = cumulative.groupby("strategy")["net_return"].transform(lambda s: (1.0 + s.fillna(0.0)).cumprod())
    fig, ax = plt.subplots(figsize=(10, 5))
    for strategy, part in cumulative.groupby("strategy"):
        ax.plot(part["month"], part["wealth"], label=strategy, linewidth=1.8)
    ax.axhline(1.0, color="black", linewidth=1)
    ax.set_title("Selected top strategies: test cumulative wealth")
    ax.set_xlabel("Month")
    ax.set_ylabel("Growth of $1")
    ax.legend(fontsize=7, loc="best")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "portfolio_top5_test_cumulative_wealth.png", dpi=160, bbox_inches="tight")
    plt.show()

## 5. Turnover Analysis

Turnover is the implementation cost channel because transaction cost is proportional to monthly trading. At 25 bps, higher-turnover strategies give up more gross alpha to rebalancing costs. Comparing turnover across model families highlights which signals are more implementable after costs, not just which rank well before trading frictions.


In [ ]:
turnover_table = selected_test[[
    "strategy", "model_family", "average_turnover", "sharpe", "annualized_return",
    "max_drawdown", "average_gross_exposure", "average_net_exposure"
]].sort_values("average_turnover")
turnover_table.to_csv(TABLE_DIR / "portfolio_top5_turnover_analysis.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4.8))
colors = ["#8C564B" if fam == "baseline_model" else "#4C78A8" for fam in turnover_table["model_family"]]
ax.barh(turnover_table["strategy"], turnover_table["average_turnover"], color=colors)
ax.set_title(f"Average monthly turnover: selected top strategies ({ANALYSIS_COST_BPS:.0f}bps)")
ax.set_xlabel("Average turnover")
for y, value in enumerate(turnover_table["average_turnover"]):
    ax.text(value, y, f" {value:.2f}", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "portfolio_top5_turnover.png", dpi=160, bbox_inches="tight")
plt.show()
turnover_table


## Financial Interpretation

Strategies with beta near zero are closer to market neutral; they mainly earn returns from cross-sectional ranking rather than broad equity market exposure. Positive beta indicates dependence on market direction, which can inflate raw returns in rising markets and increase drawdown exposure in falling markets.

Alpha is annualized from the monthly regression intercept and is computed using 25 bps net returns. The alpha t-statistic tests whether the monthly intercept is reliably different from zero. Large positive alpha with low beta is the strongest evidence of an independent stock-selection signal; weak t-statistics suggest that apparent alpha may be noisy even when realized Sharpe is attractive.

Turnover completes the implementation view. Since `transaction_cost = turnover * 25 / 10000`, a strategy with higher turnover needs more gross edge to survive costs. The most implementable strategy is therefore not necessarily the highest gross-return strategy, but the one that preserves attractive net Sharpe and alpha with moderate turnover and controlled drawdown.


## Output Manifest

In [ ]:
manifest = pd.DataFrame([
    {"artifact": "all_model_validation_ranking_25bps", "path": TABLE_DIR / "portfolio_validation_strategy_ranking_all_models.csv"},
    {"artifact": "top10_validation_sharpe_25bps", "path": TABLE_DIR / "portfolio_top10_validation_sharpe.csv"},
    {"artifact": "selected_top5_25bps", "path": TABLE_DIR / "portfolio_selected_top5_strategies.csv"},
    {"artifact": "selected_top5_test_performance_25bps", "path": TABLE_DIR / "portfolio_selected_top5_test_performance.csv"},
    {"artifact": "final_table_25bps", "path": TABLE_DIR / "portfolio_final_table_25bps.csv"},
    {"artifact": "alpha_beta_top5_25bps", "path": TABLE_DIR / "portfolio_alpha_beta_top5.csv"},
    {"artifact": "turnover_analysis_25bps", "path": TABLE_DIR / "portfolio_top5_turnover_analysis.csv"},
    {"artifact": "market_proxy", "path": TABLE_DIR / "portfolio_market_return_proxy.csv"},
    {"artifact": "heatmap", "path": FIGURE_DIR / "portfolio_top5_validation_metric_heatmap.png"},
    {"artifact": "cumulative_wealth", "path": FIGURE_DIR / "portfolio_top5_test_cumulative_wealth.png"},
    {"artifact": "turnover_plot", "path": FIGURE_DIR / "portfolio_top5_turnover.png"},
])
manifest["path"] = manifest["path"].astype(str)
manifest.to_csv(TABLE_DIR / "portfolio_analysis_notebook15_manifest.csv", index=False)
manifest
